# 09 — Preparación para espacio de datos

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

Cada algoritmo implementado queda descrito como activo publicable con endpoint, esquemas de entrada/salida y tipos de problema soportados.


In [1]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


Proyecto: /home/edmundo/packing-services


In [2]:
from _shared.loaders import SHOWCASE_INSTANCES, BENCHMARK_GROUPS, SHOWCASE_DIFFERENTIATION, load_master_catalog
from packing_services.services.algorithm_input_service import AlgorithmInputService
catalog = runners.run_dataspace_catalog()
print('dataspace_ready:', catalog.dataspace_ready)
master = load_master_catalog()
print(f'Catálogo homogéneo: {master["catalog_version"]} — {master["description"]}')
import pandas as pd
from IPython.display import display as ipy_display
agg = [s for s in catalog.services if not s.service_name.endswith('Algorithm Service')]
rows = [{'Servicio': s.service_name, 'Problema': s.problem_type, 'Algoritmos': len(s.algorithms),
         'Input': s.input_schema, 'Output': s.output_schema, 'Endpoint': s.execution_endpoint} for s in agg]
ipy_display(pd.DataFrame(rows).style.hide(axis='index'))
algo_rows = [{'Algoritmo': s.algorithms[0], 'Input': s.input_schema, 'Output': s.output_schema,
              'Endpoint': s.execution_endpoint} for s in catalog.services if s.service_name.endswith('Algorithm Service')]
print(f'Servicios por algoritmo: {len(algo_rows)}')
ipy_display(pd.DataFrame(algo_rows).style.hide(axis='index'))
bench_rows = [{'Grupo': g, 'Instancia': SHOWCASE_INSTANCES.get(g, '—'), 'Diferenciación': SHOWCASE_DIFFERENTIATION.get(g, '—')} for g in BENCHMARK_GROUPS]
ipy_display(pd.DataFrame(bench_rows).style.hide(axis='index'))


dataspace_ready: True
Catálogo homogéneo: showcase-v2 — Catálogo homogéneo P1–P5 usado en todos los notebooks didácticos. 30 unidades, 5 SKUs, mismas dimensiones en 3D-BPP, Container Loading y Single Container.


Servicio,Problema,Algoritmos,Input,Output,Endpoint
3D Bin Packing Offline Service,3D_BPP,19,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/{algorithm_name}/execute
Container Loading Service,CONTAINER_LOADING,11,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/{algorithm_name}/execute
Cartonization Service,CARTONIZATION,5,CartonizationAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/{algorithm_name}/execute
Palletization Service,PALLETIZATION,2,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/{algorithm_name}/execute
Stacking-aware Packing Service,STACKING_AWARE,2,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/{algorithm_name}/execute
Packing Validation Service,VALIDATION,1,ValidateRequest,ValidateResponse,POST /api/v1/validate
Packing Benchmark / Comparison Service,BENCHMARK,19,BenchmarkRequest,BenchmarkResponse,POST /api/v1/benchmark


Servicios por algoritmo: 35


Algoritmo,Input,Output,Endpoint
boxpacker_adapter,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/boxpacker_adapter/execute
container_packing_adapter,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/container_packing_adapter/execute
dwave_adapter,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/dwave_adapter/execute
packingsolver_adapter,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/packingsolver_adapter/execute
py3dbp_adapter,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/py3dbp_adapter/execute
skjolber_adapter,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/skjolber_adapter/execute
aco_3d_bpp,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/aco_3d_bpp/execute
best_box_volume_utilization,CartonizationAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/best_box_volume_utilization/execute
best_fit_decreasing_3d,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/best_fit_decreasing_3d/execute
bin_reduction,PackAlgorithmInput,AlgorithmExecuteResponse,POST /api/v1/algorithms/bin_reduction/execute


Grupo,Instancia,Diferenciación
3D_BPP,showcase_3d_bpp_instance,best_fit ~27 emp. (85% util) vs extreme_points ~22 (66%)
3D_HYBRID,showcase_3d_bpp_instance,misma cantidad empacada; compaction/LS mueven 8–11 piezas
3D_IMPROVEMENT,showcase_3d_bpp_instance,misma baseline; relocation/swap/orientation/bin_reduction refinan layout
3D_METAHEURISTIC,showcase_3d_bpp_instance,metaheurísticas optimizan orden de colocación sobre baseline constructivo
CONTAINER_LOADING,showcase_container_loading_instance,weight_aware ~27 emp. vs wall_building ~24 vs single_container ~17
CARTONIZATION,showcase_cartonization_instance,single-box → BOX_M (~49%) vs multi_box → varias cajas S/M
SINGLE_CONTAINER_LOADING,showcase_single_container_instance,best_fit/single ~13 emp. (94% util) vs first_fit ~16 (89%)
PALLETIZATION,showcase_palletization_instance,layer_based ~capas horizontales vs stack_based ~columnas
STACKING_AWARE,showcase_stacking_aware_instance,stacking_aware respeta soporte/carga vs stack_based libre


In [3]:
svc = AlgorithmInputService()
example = svc.build_input_example('multi_box_cartonization')
print('Ejemplo de entrada (CartonizationAlgorithmInput):', sorted(example.keys()))
result = runners.run_algorithm_execute('multi_box_cartonization', example)
display.show_execute_result(result)
print('Cajas usadas:', result.cartonization.selected_box_ids if result.cartonization else '—')


2026-07-13 12:47:09,780 | INFO    | packing_services.services.algorithm_execution_service | algorithm-execute name=multi_box_cartonization problem_type=CARTONIZATION request_id=carton-001


2026-07-13 12:47:09,781 | INFO    | packing_services.services.cartonization_service | cartonization request_id=carton-001 algorithm=multi_box_cartonization items=5 boxes=3


Ejemplo de entrada (CartonizationAlgorithmInput): ['boxes', 'constraints', 'items', 'parameters', 'request_id']


Cajas usadas: ['BOX_S#1', 'BOX_S#2']


## Cierre del recorrido didáctico

1. **Cuatro tipos de problema** con instancia showcase dedicada (`examples/showcase_*_instance.json`)
2. **Veintidós algoritmos** implementados + adaptador ``py3dbp`` (si está instalado)
3. **Ocho benchmarks** comparativos: 3D-BPP, híbrido, mejora 3D, CL, Cartonization, SCL, Palletization, Stacking-aware
4. **Perfiles estándar** (`constructive`, `hybrid`, `box_selection`) para comparación reproducible
5. **Validación independiente** y **descriptores** listos para espacio de datos (`GET /api/v1/services`)
6. **Un endpoint por algoritmo** con salida `AlgorithmExecuteResponse` estandarizada

Cada benchmark usa un caso diseñado para **diferenciar** estrategias del mismo grupo — evidencia cuantitativa del avance del proyecto.
